# LifeMate Qwen3-4B next-action QLoRA

Uses the fixed train-only, replayed trajectory dataset. The smoke run trains on 128 examples for 30 optimizer steps with gradient accumulation 4, then reloads the saved adapter and probes it with a real SFT trajectory prompt.

Keep SMOKE_RUN=True until the smoke adapter is saved, reloaded, and its action output parses. Set it to False only for full training.


In [ ]:
import os, subprocess, sys
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'MISSING')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.51.3', 'peft==0.15.2', 'bitsandbytes==0.45.5', 'accelerate==1.7.0'])


In [ ]:
from pathlib import Path
SMOKE_RUN = True
input_root = Path('/kaggle/input')
print('Kaggle inputs:', [p.name for p in input_root.iterdir()])
trainer_path = next(input_root.rglob('train_qwen_next_action.py'), None)
if trainer_path is None:
    raise FileNotFoundError('train_qwen_next_action.py is missing from attached Kaggle datasets')
data_path = trainer_path.with_name('sft_train.jsonl')
if not data_path.is_file():
    raise FileNotFoundError(f'SFT JSONL missing beside runner: {data_path}')
print('Training input directory:', trainer_path.parent)
args = [sys.executable, str(trainer_path), '--data', str(data_path),
        '--output', '/kaggle/working/qwen-next-action-lora-v1', '--epochs', '2', '--max-length', '4096']
if SMOKE_RUN:
    args += ['--smoke', '--smoke-examples', '128', '--max-steps', '30']
env = os.environ.copy()
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
subprocess.check_call(args, env=env)
